In [0]:
gold_df = spark.table("lms_analytics.silver_enriched_enrolments")

print("Silver data loaded successfully")
print("Rows:", gold_df.count())

In [0]:
gold_df.printSchema()

In [0]:
display(gold_df)

In [0]:
print("Columns in Silver table:")
for col in gold_df.columns:
    print(col)

In [0]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS lms_analytics
""")

print("Gold layer will use database: lms_analytics")

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    avg,
    sum,
    round,
    when
)

course_performance_df = gold_df.groupBy(
    "course_id",
    "course_title",
    "category",
    "instructor_id",
    "instructor_name",
    "difficulty_level"
).agg(
    count("enrolment_id").alias("total_enrolments"),
    round(avg("progress_pct"), 2).alias("avg_progress_pct"),
    sum(when(col("status") == "Completed", 1).otherwise(0)).alias("completed_learners"),
    sum(when(col("status") == "In Progress", 1).otherwise(0)).alias("in_progress_learners"),
    sum(when(col("status") == "Not Started", 1).otherwise(0)).alias("not_started_learners")
)

display(course_performance_df)

In [0]:
course_performance_df = course_performance_df.withColumn(
    "completion_rate_pct",
    round(
        (col("completed_learners") / col("total_enrolments")) * 100,
        2
    )
)

display(course_performance_df)

In [0]:
course_performance_df = course_performance_df.withColumn(
    "performance_category",
    when(col("completion_rate_pct") >= 80, "Excellent")
    .when(col("completion_rate_pct") >= 60, "Good")
    .when(col("completion_rate_pct") >= 40, "Average")
    .otherwise("Needs Improvement")
)

display(course_performance_df)

In [0]:
course_performance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lms_analytics.gold_course_performance")

print("gold_course_performance table created successfully")

In [0]:
spark.sql("""
SELECT *
FROM lms_analytics.gold_course_performance
ORDER BY completion_rate_pct DESC
""").show(20, truncate=False)

In [0]:
learner_performance_df = gold_df.groupBy(
    "learner_id",
    "learner_name",
    "email",
    "city",
    "subscription_type"
).agg(
    count("enrolment_id").alias("total_courses"),
    round(avg("progress_pct"), 2).alias("avg_progress_pct"),
    sum(when(col("status") == "Completed", 1).otherwise(0)).alias("completed_courses"),
    sum(when(col("status") == "In Progress", 1).otherwise(0)).alias("in_progress_courses"),
    sum(when(col("status") == "Not Started", 1).otherwise(0)).alias("not_started_courses")
)

display(learner_performance_df)

In [0]:
learner_performance_df = learner_performance_df.withColumn(
    "completion_rate_pct",
    round(
        (col("completed_courses") / col("total_courses")) * 100,
        2
    )
)

display(learner_performance_df)

In [0]:
learner_performance_df = learner_performance_df.withColumn(
    "performance_category",
    when(col("completion_rate_pct") >= 80, "Excellent")
    .when(col("completion_rate_pct") >= 60, "Good")
    .when(col("completion_rate_pct") >= 40, "Average")
    .otherwise("Needs Improvement")
)

display(learner_performance_df)

In [0]:
learner_performance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lms_analytics.gold_learner_performance")

print("gold_learner_performance table created successfully")

In [0]:
spark.sql("""
SELECT *
FROM lms_analytics.gold_learner_performance
ORDER BY completion_rate_pct DESC
""").show(20, truncate=False)

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    avg,
    round,
    sum,
    when
)

enrollment_analytics_df = gold_df.groupBy(
    "course_id",
    "course_title",
    "category"
).agg(
    count("enrolment_id").alias("total_enrolments"),
    countDistinct("learner_id").alias("unique_learners"),
    round(avg("progress_pct"), 2).alias("avg_progress_pct"),
    sum(
        when(col("status") == "Completed", 1).otherwise(0)
    ).alias("completed_enrolments"),
    sum(
        when(col("status") == "In Progress", 1).otherwise(0)
    ).alias("active_enrolments"),
    sum(
        when(col("status") == "Not Started", 1).otherwise(0)
    ).alias("not_started_enrolments")
)

display(enrollment_analytics_df)

In [0]:
enrollment_analytics_df = enrollment_analytics_df.withColumn(
    "completion_rate_pct",
    round(
        (col("completed_enrolments") /
         col("total_enrolments")) * 100,
        2
    )
)

display(enrollment_analytics_df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

enrollment_window = Window.orderBy(
    col("total_enrolments").desc()
)

enrollment_analytics_df = enrollment_analytics_df.withColumn(
    "enrollment_rank",
    row_number().over(enrollment_window)
)

display(enrollment_analytics_df)

In [0]:
enrollment_analytics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lms_analytics.gold_enrollment_analytics")

print("gold_enrollment_analytics table created successfully")

In [0]:
spark.sql("""
SELECT
    enrollment_rank,
    course_id,
    course_title,
    category,
    total_enrolments,
    unique_learners,
    avg_progress_pct,
    completion_rate_pct
FROM lms_analytics.gold_enrollment_analytics
ORDER BY enrollment_rank
""").show(20, truncate=False)

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    avg,
    round,
    sum,
    when,
    max as spark_max
)

learning_activity_df = gold_df.groupBy(
    "learner_id",
    "learner_name"
).agg(
    count("enrolment_id").alias("total_enrolments"),
    round(avg("progress_pct"), 2).alias("avg_progress_pct"),
    spark_max("last_activity_date").alias("last_activity_date"),
    sum(
        when(col("status") == "Completed", 1).otherwise(0)
    ).alias("completed_courses"),
    sum(
        when(col("status") == "In Progress", 1).otherwise(0)
    ).alias("active_courses")
)

display(learning_activity_df)

In [0]:
from pyspark.sql.functions import datediff, current_date

learning_activity_df = learning_activity_df.withColumn(
    "days_since_last_activity",
    datediff(current_date(), col("last_activity_date"))
)

learning_activity_df = learning_activity_df.withColumn(
    "activity_status",
    when(col("days_since_last_activity") <= 7, "Active")
    .when(col("days_since_last_activity") <= 30, "Moderately Active")
    .otherwise("Inactive")
)

display(learning_activity_df)

In [0]:
learning_activity_df = learning_activity_df.withColumn(
    "learner_risk",
    when(
        (col("avg_progress_pct") < 40) &
        (col("activity_status") == "Inactive"),
        "High Risk"
    )
    .when(
        (col("avg_progress_pct") < 60) |
        (col("activity_status") == "Moderately Active"),
        "Medium Risk"
    )
    .otherwise("Low Risk")
)

display(learning_activity_df)

In [0]:
learning_activity_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lms_analytics.gold_learning_activity")

print("gold_learning_activity table created successfully")

In [0]:
spark.sql("""
SELECT
    learner_id,
    learner_name,
    total_enrolments,
    avg_progress_pct,
    last_activity_date,
    days_since_last_activity,
    activity_status,
    learner_risk
FROM lms_analytics.gold_learning_activity
ORDER BY
    learner_risk DESC,
    days_since_last_activity DESC
""").show(20, truncate=False)

In [0]:
spark.sql("""
SHOW TABLES IN lms_analytics
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT
    learner_id,
    COUNT(*) AS record_count
FROM lms_analytics.gold_learner_performance
GROUP BY learner_id
HAVING COUNT(*) > 1
""").show()

In [0]:
spark.sql("""
SELECT
    course_id,
    COUNT(*) AS record_count
FROM lms_analytics.gold_course_performance
GROUP BY course_id
HAVING COUNT(*) > 1
""").show()

In [0]:
print("Gold Course Performance:", 
      spark.table("lms_analytics.gold_course_performance").count())

print("Gold Learner Performance:", 
      spark.table("lms_analytics.gold_learner_performance").count())

print("Gold Enrollment Analytics:", 
      spark.table("lms_analytics.gold_enrollment_analytics").count())

print("Gold Learning Activity:", 
      spark.table("lms_analytics.gold_learning_activity").count())